Imlo coursework

In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
# code to use my gpu
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(device)

cuda


In [3]:
#transform to the train dataset with augmentation
train_transform = transforms.Compose([
    # adding random augmentations
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),

    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# transforms for the val dataset without data augmentations
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [4]:
#defining the train dataset
train_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = train_transform,
    download = True
)

#defining the val dataset
val_data = torchvision.datasets.OxfordIIITPet(
    root = './data',
    split = 'trainval',
    transform = val_transform,
    download = False
)

#split will be 80:20
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size


train_indices, val_indices = torch.utils.data.random_split(
    range(len(train_data)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42) #makes sure that the images used in train and val stick tg
)

train_data = torch.utils.data.Subset(
    train_data,
    train_indices.indices
)

val_data = torch.utils.data.Subset(
    val_data,
    val_indices.indices
)

#dataloaders for train and val
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=64, shuffle=False, num_workers=2)

100%|██████████| 792M/792M [00:29<00:00, 26.7MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 9.94MB/s]


In [5]:
image, label = train_data[0]

In [6]:
image.size()

torch.Size([3, 128, 128])

In [7]:
# Adding names for the catergories
class_names = train_data.dataset.classes

In [38]:
# Defining the layers
class NeuralNet(nn.Module):
    def __init__(self):
    #calls constructor from nn.Module
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        #self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4)) - trying without adaptive pooling

        #self.dropout = nn.Dropout(0.5)

        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 37)

    def forward(self, input):
        input = F.relu(self.conv1(input)) #conv1 then relu
        input = self.pool(input) #1st max pool

        input = F.relu(self.conv2(input)) #conv2 then relu
        input = self.pool(input) #2nd max pool

        input = F.relu(self.conv3(input)) #conv3 then relu
        input = F.relu(self.conv4(input)) #conv4 then relu
        input = self.pool(input) #3rd max pool
        #input = self.adaptive_pool(input) #adaptive pool

        input = torch.flatten(input, 1)  #flattening
        input = F.relu(self.fc1(input))  #applying fc1, then RELU
        input = F.relu(self.fc2(input))  #applying fc2, then RELU
        #input = self.dropout(input)  #applying dropout
        input = self.fc3(input)  #applying fc3
        return input

In [39]:
# defining the NN itself
network = NeuralNet().to(device)
loss_func = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(network.parameters(), lr=0.003)

In [40]:
# training the model
for epoch in range(30):
    print("Training epoch:", epoch)
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimiser.zero_grad()

        outputs = network(inputs)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()

    running_loss_calc = running_loss / len(train_loader)
    print("Loss:", running_loss_calc)

Training epoch: 0
Loss: 3.589529379554417
Training epoch: 1
Loss: 3.4498353678247202
Training epoch: 2
Loss: 3.35340880829355
Training epoch: 3
Loss: 3.2804248695788174
Training epoch: 4
Loss: 3.214772582054138
Training epoch: 5
Loss: 3.1249121272045635
Training epoch: 6
Loss: 3.01646379802538
Training epoch: 7
Loss: 2.9503768475159355
Training epoch: 8
Loss: 2.8089367773221885
Training epoch: 9
Loss: 2.69956214013307
Training epoch: 10
Loss: 2.585537910461426
Training epoch: 11
Loss: 2.4515974366146587
Training epoch: 12
Loss: 2.330411644085594
Training epoch: 13
Loss: 2.1801917941673943
Training epoch: 14
Loss: 2.0259688548419788
Training epoch: 15
Loss: 1.899857601393824
Training epoch: 16
Loss: 1.7902244329452515
Training epoch: 17
Loss: 1.6126219552496206
Training epoch: 18
Loss: 1.5600518143695334
Training epoch: 19
Loss: 1.4112655960995217
Training epoch: 20
Loss: 1.2629568226959393
Training epoch: 21
Loss: 1.1270742649617402
Training epoch: 22
Loss: 1.089189534601958
Training e

In [41]:
# testing the model on val data
correct = 0
total = 0

network.eval()

with torch.no_grad():
  for images, labels in val_loader:

    images = images.to(device)
    labels = labels.to(device)


    outputs = network(images)
    predicted = outputs.argmax(1)

    total += len(labels)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print("Accuracy:", accuracy)

Accuracy: 19.565217391304348
